In [ ]:
!apt update && apt install -y openslide-tools
!pip install openslide-python
!pip install tiatoolbox

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [62.5 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,234 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd '/content/drive/My Drive/2_PHDWork/1_Objective-I/4_Enlargement_Strategy/'

/content/drive/My Drive/2_PHDWork/1_Objective-I/4_Enlargement_Strategy


In [ ]:
from openslide import OpenSlide
import os
import matplotlib.pyplot as plt
import numpy as np
import cv2 as cv
import time
import csv
import math

from shapely import get_coordinates, concave_hull, buffer, LineString, Point
from shapely.geometry import Polygon
from shapely.affinity import translate
from shapely.ops import unary_union
from shapely.geometry.multipolygon import MultiPolygon

# Ostu tissue mask generator.
from tiatoolbox.tools.tissuemask import OtsuTissueMasker

2025-01-21 08:31:17.563153: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-21 08:31:17.940758: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-21 08:31:18.045326: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-21 08:31:18.701787: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-21 08:31:22.653717: W tensorflow/compiler/tf2

In [ ]:
"""
The path and name of the log file to which comparison data is writted.

Note: Set it to appropriate filepath and set the 'enable_log' to True for
      writing comparison data.
"""
log_file_name = './Enlargment_FIMG_log_data_Level_0_T3_00.csv'

"""
Writes the comparison data to a csv file.

Parameters:
     log_data: the comparison data to be written.

Returns:
     None
"""
def log_comparison_data(log_data):
  fields = ['dataset','f_name','t_row','t_col','i_row','i_col','read_time',
            's_cntr_trans_time','s_msk_fill_time','m_cntr_trans_time',
            'm_msk_fill_time','org_msk_gen_time','s_mem_req','m_mem_req',
            'o_mem_req','s_accuracy','s_dice','s_iou','s_hdit','s_tpr','s_fpr',
            'm_accuracy','m_dice','m_iou','m_hdit','m_tpr','m_fpr']

  # Creating a log file if it does not exist.
  if(not os.path.isfile(log_file_name)):
    with open(log_file_name, 'w') as csvfile:
      # creating a csv writer object
      csvwriter = csv.writer(csvfile)
      # writing the fields
      csvwriter.writerow(fields)

  # Writing comparison data to the file.
  try:
    with open(log_file_name,"a") as csvFile:
      writer = csv.DictWriter(csvFile, fieldnames=fields)
      writer.writerow(log_data)
  except IOError:
    print("Writing of Training data failed")

  return

In [ ]:
"""
  Method used to scale the polygon co-ordinates.
"""
def scale_polygon_coordinates(polygon,s_r,s_c):
  coordinates = get_coordinates(polygon)
  coordinates[:,0] *= s_c
  coordinates[:,1] *= s_r

  s_polygon = Polygon(coordinates)

  return s_polygon

"""
  Method to convert the contours to displayable contours.
"""
def convert_to_displayable_contours(contours,max_r,max_c):
  dis_cont = []
  for i in range(len(contours)):
    cnt = np.ceil(contours[i].copy()).astype(np.int32)
    cnt[:,:,0] = np.clip(cnt[:,:,0],0,max_c)
    cnt[:,:,1] = np.clip(cnt[:,:,1],0,max_r)
    dis_cont.append(cnt)

  return dis_cont

In [ ]:
"""
  The method performing the simple scaling of the contour points.
"""
def get_equation_based_tissue_mask(contours,t_size,l_size,log_data):
  s_r = l_size[0] / t_size[0]
  s_c = l_size[1] / t_size[1]
  bin_mask = np.zeros(l_size,dtype=np.uint8)
  new_points = []

  # Computing on the approximated contours
  start_time = time.process_time()
  for pnts in contours:
    wrk_pnts = pnts.astype(np.float32)
    wrk_pnts[:,:,0] = wrk_pnts[:,:,0] * s_c
    wrk_pnts[:,:,1] = wrk_pnts[:,:,1] * s_r
    wrk_pnts = (np.floor(wrk_pnts)).astype(np.int32)
    new_points.append(wrk_pnts)
  end_time = time.process_time()
  cntr_trans_time = (end_time - start_time)
  log_data['s_cntr_trans_time'] = cntr_trans_time

  start_time = time.process_time()
  bin_mask = cv.fillPoly(bin_mask, new_points, (255))
  end_time = time.process_time()
  msk_fill_time = (end_time - start_time)
  log_data['s_msk_fill_time'] = msk_fill_time

  mem_req = 0
  for pnts in contours:
    total_points = 0
    for i in range(pnts.shape[0]):
      for j in range(pnts.shape[1]):
        total_points += 2
    mem_req +=  total_points * 4

  log_data['s_mem_req'] = mem_req

  return bin_mask,log_data,new_points

In [ ]:
par = 0.5

"""
Generates the high-resolution tissue mask from the correspodning low-resolution
tissue co-ordinates.

Parameters:
     o_contours:  the low-resolution tissue co-ordinates
     t_size    :  low-resolution image size.
     l_size    :  high-resolution image size.
     log_data  :  the log data to be written.

Returns:
     bin_mask   : the generated high-resolution tissue mask
     log_data   : the log data to be written.
     new_points : the high-resolution tissue co-ordinates.
     output_img : the image with the marked tissue boundaries.
"""
def get_enlargment_based_tissue_mask2(o_contours,t_size,l_size,log_data):
  s_r = l_size[0] / t_size[0]
  s_c = l_size[1] / t_size[1]
  i_size = (l_size[0],l_size[1],3)
  output_img = np.zeros(i_size,dtype=np.uint8)
  bin_mask = np.zeros(l_size,dtype=np.uint8)
  new_points = []
  o_s_points = []
  # t_s_points = []

  # Step-1: Transformation of co-ordinates.
  start_time = time.process_time()
  for i in range(len(o_contours)):
    cnt = np.reshape(o_contours[i].copy(),(-1,2)).astype(np.float32)

    if(cnt.shape[0] <= 10):
      if(cnt.shape[0] == 1):
        # cnt = np.append(cnt,[[cnt[0,0]+1,cnt[0,1]+1],[cnt[0,0]+2,cnt[0,1]+2]],axis=0)
        poly1 = buffer(Point(cnt), 0.3,join_style='mitre')
      else:
        # cnt = np.append(cnt,[[cnt[1,0]+1,cnt[1,1]+1]],axis=0)
        poly1 = buffer(LineString(cnt.tolist()), 0.3,join_style='mitre')

      comb_ply = scale_polygon_coordinates(poly1,s_r,s_c)
      if not comb_ply.is_valid:
        comb_ply = comb_ply.buffer(0.1,join_style='mitre')
    else:
      poly1 = Polygon(cnt)
      poly2 = translate(poly1, xoff=par, yoff=par)    # Bottom-right
      poly3 = translate(poly1, xoff=-par, yoff=par)   # Bottom-left
      poly4 = translate(poly1, xoff=par, yoff=-par)   # Top-right

      poly1 = scale_polygon_coordinates(poly1,s_r,s_c)
      poly2 = scale_polygon_coordinates(poly2,s_r,s_c)
      poly3 = scale_polygon_coordinates(poly3,s_r,s_c)
      poly4 = scale_polygon_coordinates(poly4,s_r,s_c)

      if not poly1.is_valid:
        poly1 = poly1.buffer(0.1,join_style='mitre')
      if not poly2.is_valid:
        poly2 = poly2.buffer(0.1,join_style='mitre')
      if not poly3.is_valid:
        poly3 = poly3.buffer(0.1,join_style='mitre')
      if not poly4.is_valid:
        poly4 = poly4.buffer(0.1,join_style='mitre')

      comb_ply = unary_union([poly1,poly2,poly3,poly4])

      if isinstance(comb_ply, MultiPolygon):
        comb_ply = concave_hull(comb_ply,ratio=0.001)

    new_points.append(np.reshape(np.array(list(comb_ply.exterior.coords)),(-1,1,2)))
    o_s_points.append(np.reshape(np.array(list(poly1.exterior.coords)),(-1,1,2)))
    # t_s_points.append(np.reshape(np.array(list(poly2.exterior.coords)),(-1,1,2)))

  end_time = time.process_time()
  cntr_trans_time = (end_time - start_time)
  log_data['m_cntr_trans_time'] = cntr_trans_time

  o_s_points = convert_to_displayable_contours(o_s_points,l_size[0],l_size[1])
  # t_s_points = convert_to_displayable_contours(t_s_points,l_size[0],l_size[1])
  new_points = convert_to_displayable_contours(new_points,l_size[0],l_size[1])

  # Step-2: Generating the mask.
  start_time = time.process_time()
  bin_mask = cv.fillPoly(bin_mask, new_points, (255))
  end_time = time.process_time()
  msk_fill_time = (end_time - start_time)
  log_data['m_msk_fill_time'] = msk_fill_time

  r_mem_req = 0
  for pnts in new_points:
    total_points = 0
    for i in range(pnts.shape[0]):
      for j in range(pnts.shape[1]):
        total_points += 2
    r_mem_req +=  total_points * 4
  log_data['m_mem_req'] = r_mem_req
  log_data['o_mem_req'] = l_size[0] * l_size[1] * 3

  cv.drawContours(output_img, o_s_points, -1, (255,0,0), 1)
  # cv.drawContours(output_img, t_s_points, -1, (255,0,0), 1)
  cv.drawContours(output_img, new_points, -1, (0,255,0), 1)

  return bin_mask,log_data,new_points,output_img

In [ ]:
"""
Performs a pixel to pixel matching between the given masks and returns the
accuracy, Dice's and Jaccard's coefficient.

Parameters:
     org:     the original labels.

     img2:    the predicted labels.

Returns:
     accuracy:    the percentage of pixels that matches.

     dice_p:      the Dice's coefficients for tissue pixels.

     jaccard_p:   the Jaccard's coefficients for tissue pixels.

     dice_n:      the Dice's coefficients for non-tissue pixels.

     jaccard_n:   the Jaccard's coefficients for non-tissue pixels.

"""
from scipy.spatial.distance import directed_hausdorff

def measure_performance_metrics(org,pred):
  # Converting the values into 0 and 1.
  i1 = org // 255
  i2 = pred // 255

  # Calculating the TP, TN, FP and FN
  TP = np.sum((i1 * i2))
  TN = np.sum((1 - i1) * (1 - i2))
  FP = np.sum((((i2 - i1) + 1)//2))
  FN = np.sum((((i1 - i2) + 1)//2))

  accuracy = round(((TP + TN)/(TP + TN + FP + FN)),6) * 100

  area_intersection = TP
  area_union = TP + FN + FP

  if(area_union != 0):
    dice = 2 * area_intersection / (np.sum(i1) + np.sum(i2))
    iou = area_intersection / area_union
    # jaccard = area_intersection / area_union
  else:
    dice = 0
    iou = 0

  # Find contours (boundary points) of the masks
  contours1, _ = cv.findContours(org, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)
  contours2, _ = cv.findContours(pred, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)

  # Extract boundary points
  points1 = np.vstack(contours1).squeeze()
  points2 = np.vstack(contours2).squeeze()

  u = directed_hausdorff(points1, points2)[0]
  v = directed_hausdorff(points2, points1)[0]
  hausdorff_distance = max(u, v)
  # hausdorff_distance = 0

  TPR = round((TP / (TP + FN)),8)
  FPR = round((FP / (FP + TN)),8)

  return (accuracy,dice,iou,hausdorff_distance,TPR,FPR)

In [ ]:
"""
  Saves the generated images to drive.
"""
def save_result_images(wsi,m_name,ext,index,level,t_mask,gen_mask,o_s_points,n_cntrs):
  # t_level =  wsi.level_count-1
  # t_image = cv.cvtColor(np.array(wsi.read_region((0,0),t_level,wsi.level_dimensions[t_level])),cv.COLOR_RGBA2RGB)
  # file_name = "./Otsu_Images/"+ str(index) + "_" +m_name + "_Thumbnail" + ".png"
  # plt.imshow(t_image)
  # plt.savefig(file_name,format='png',bbox_inches='tight',pad_inches = 0)
  # plt.clf()
  # del t_image

  t_image = cv.cvtColor(np.array(wsi.read_region((0,0),level,wsi.level_dimensions[level])),cv.COLOR_RGBA2RGB)
  # file_name = "./Otsu_Images/"+ str(index)+ "_"  + m_name + "_Level_"+ str(level) + ".png"
  # plt.imshow(t_image)
  # plt.savefig(file_name,format='png',bbox_inches='tight',pad_inches = 0)
  # plt.clf()

  cv.drawContours(t_image, n_cntrs, -1, (0,255,0), 10)
  file_name = "./Otsu_Images/"+ str(index)+ "_"  + m_name + "_Contour_Output_" + str(level) + ".png"
  file_name2 = "./Otsu_Images/"+ str(index)+ "_"  + m_name + "_Contour_Output_Img" + str(level) + ".png"
  cv.imwrite(file_name2,t_image)
  time.sleep(6)
  t_image = cv.imread(file_name2)
  plt.imshow(t_image)
  plt.savefig(file_name,format='png',bbox_inches='tight',pad_inches = 0)
  plt.clf()
  del t_image

  # Writing the thumbnail mask images to drive.
  # file_name = "./Otsu_Images/"+ str(index)+ "_"  + m_name + "_Thumbnail_Mask" + ".png"
  # plt.imshow(t_mask,cmap='gray')
  # plt.savefig(file_name,format='png',bbox_inches='tight',pad_inches = 0)
  # plt.clf()

  # Writing the generated mask images to drive.
  # file_name = "./Otsu_Images/"+ str(index)+ "_"  + m_name + "_Generated_Mask_"+ str(level) + ".png"
  # plt.imshow(gen_mask,cmap='gray')
  # plt.savefig(file_name,format='png',bbox_inches='tight',pad_inches = 0)
  # plt.clf()

  print(f'\tAll images written to drive.')

  return

In [ ]:
"""
  Generates the Otsu tissue mask for the give image using standard library API.
"""
def get_otsu_tissue_mask(image):
  t_size = image.shape
  image = np.reshape(image,(1,t_size[0],t_size[1],t_size[2]))
  masker = OtsuTissueMasker()
  masks = masker.fit_transform(image) # Here we get a list of gray scale image
  true_index = np.asarray(masks[0] == True).nonzero()
  bin_mask = np.zeros(masks[0].shape,dtype=np.uint8)
  bin_mask[true_index] = 255

  return bin_mask

In [ ]:
"""
  The entry function. The execution starts here.
"""
def main_fun(path,dataset,maskpath,level = 1,enable_log = True):
  if(os.path.isdir(path)):
    files_names = os.listdir(path)

    print(f'Processing a total of {len(files_names)} WSI for Level-{level}')

    index = 1
    for file in files_names:
      wsiname = path + file
      log_data = {}
      log_data['dataset'] = dataset
      log_data['f_name'] = file

      m_name, ext = os.path.splitext(file)
      outfile = "./Output_Images/"+ str(index) + m_name + "_Contour_Output_" + str(level) + ".png"

      wsi = OpenSlide(wsiname)

      # Populating the details of the WSI
      t_level =  wsi.level_count-1
      l_size = (wsi.level_dimensions[level][1],wsi.level_dimensions[level][0])
      t_size = (wsi.level_dimensions[t_level][1],wsi.level_dimensions[t_level][0])
      log_data['t_row'] = t_size[0] # Thumbnail Size
      log_data['t_col'] = t_size[1] # Thumbnail Size
      log_data['i_row'] = l_size[0] # Required Mask Size
      log_data['i_col'] = l_size[1] # Required Mask Size

      print(f'\tProcessing WSI File {index}: {file} s_r:{l_size[0]/t_size[0]} s_c:{l_size[1]/t_size[1]}')

      # Generating the thumbnail mask
      t_mask = np.zeros((t_size[0],t_size[1]),dtype=np.uint8)
      start_time = time.process_time()
      temp = np.array(wsi.read_region((0,0),t_level,wsi.level_dimensions[t_level]))
      t_image = cv.cvtColor(temp,cv.COLOR_RGBA2RGB)
      t_mask = get_otsu_tissue_mask(t_image)

      o_contours, hierarchy = cv.findContours(t_mask,cv.RETR_EXTERNAL,
                                              cv.CHAIN_APPROX_NONE)
      end_time = time.process_time()
      read_time = (end_time - start_time)
      log_data['read_time'] = read_time

      s_mask,log_data,o_s_points = get_equation_based_tissue_mask(o_contours,t_size,
                                                              l_size,log_data)

      # Generating enlarged new mask and the corresponding contours
      mask,log_data,n_cntrs,output_img= get_enlargment_based_tissue_mask2(o_contours,t_size,
                                                              l_size,log_data)

      del t_image
      del temp
      del o_contours
      del hierarchy
      del s_mask

      save_result_images(wsi,m_name,ext,index,level,t_mask,mask,o_s_points,n_cntrs)

      del output_img
      del wsi
      del mask
      del n_cntrs
      del t_mask
      del o_s_points

      # if(enable_log):
      #   log_comparison_data(log_data)

      index += 1
      # break
      # if(index == 5):
      #   break
  return

path = "./CPTAC-CM-Dataset/" # Path to the WSI.
mskpath = ""
main_fun(path,'CPTAC-CM',mskpath,level = 0,enable_log = False)

Processing a total of 4 WSI for Level-0
	Processing WSI File 1: C3L-02102-27.svs s_r:8.001345895020188 s_c:8.000436871996506
	All images written to drive.
	Processing WSI File 2: C3L-00275-21.svs s_r:8.000806451612902 s_c:8.002163164400494
	All images written to drive.
	Processing WSI File 3: C3L-00275-23.svs s_r:16.003205128205128 s_c:16.005113221329438
	All images written to drive.
	Processing WSI File 4: C3L-00629-23.svs s_r:8.003543586109142 s_c:8.003125
	All images written to drive.


<Figure size 640x480 with 0 Axes>